In [3]:
!pip install numpy scipy biopython torch
import numpy as np
from Bio.PDB import PDBParser
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
import sys
sys.path.append('scripts')

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
import pdb_to_graph
import mldft_surrogate
import verify_twin_pipeline
import pdb_voxelizier
import cnn_mlp_encoder
import jw_quantum_mapper


In [5]:
protein = "proteins/1I3V.pdb"

In [6]:
# Run track A (CNN)
tensor = pdb_voxelizier.pdb_to_tensor(protein, grid_size=32)
cnn_coefficients = cnn_mlp_encoder.get_hamiltonian(tensor)

In [8]:
print(cnn_coefficients)

[ 0.04579625 -0.04431609  0.01413401  0.0604613   0.05113716 -0.07733478
  0.0294201   0.06623122 -0.05403912  0.04147597]


In [9]:
# 1. Run Track B (ML-DFT)
graph_data = pdb_to_graph.pdb_to_graph(protein, distance_threshold=5.0)
mldft_coefficients = mldft_surrogate.get_mldft_hamiltonian(graph_data, num_qubits=4)


In [11]:
print(mldft_coefficients)

[-0.12920389  0.3636884   0.38954073 -0.40110368 -0.50586045  0.02328536
 -0.10575376 -0.50672066  0.04505328 -0.03886317]


In [12]:
# 2. Run the Verification (Assuming you have 'cnn_coefficients' from Track A)
# Use dummy data here just to test the script if needed:
#cnn_coefficients = mldft_coefficients + np.random.normal(0, 0.01, 10)
verify_twin_pipeline.cross_verify_pipelines(cnn_coefficients, mldft_coefficients, num_sites=4)


=== TWIN PIPELINE VERIFICATION REPORT ===

[Checkpoint 1] Coefficient Mean Absolute Error (MAE): 0.296515 eV
-> Status: WARNING (Check spatial mapping drift)

[Checkpoint 2] Physical Ground State Energy (E0)
Track A (3D CNN) E0 : -0.155720 eV
Track B (ML-DFT) E0 : -0.555157 eV
Delta E (Error)     : 0.399436 eV
-> Status: FAIL (Exceeds Chemical Accuracy. Do not send to QPU.)
